In [0]:
import os, json, requests

FOLDER = "/Volumes/workspace/heatcheck/raw/reference"
os.makedirs(FOLDER, exist_ok=True)
HEADERS = {"x-api-key": dbutils.secrets.get("heatcheck", "datagov_api_key")}
DATASETS = {
    "subzones": "d_8594ae9ff96d0c708bc2af633048edfb",
    "eldercare": "d_f0fd1b3643ed8bd34bd403dedd7c1533",
}

for name, ds in DATASETS.items():
    meta = requests.get(
        f"https://api-open.data.gov.sg/v1/public/api/datasets/{ds}/poll-download",
        headers=HEADERS, timeout=30).json()
    if meta.get("code") != 0:
        print(name, "FAILED:", meta)
        continue
    r = requests.get(meta["data"]["url"], timeout=120)
    r.raise_for_status()
    with open(f"{FOLDER}/{name}.geojson", "wb") as f:
        f.write(r.content)
    print("Saved", name, len(r.content) // 1024, "KB")

In [0]:
pop = spark.read.option("header", True).csv("/Volumes/workspace/heatcheck/raw/singstat/")
print(pop.columns)
display(pop.limit(10))

In [0]:
eld = json.load(open(f"{FOLDER}/eldercare.geojson"))
print(len(eld["features"]), "locations")
print(json.dumps(eld["features"][0]["properties"], indent=2)[:1500])

In [0]:
from pyspark.sql import functions as F

pop_clean = (pop
    .withColumn("pop", F.expr("try_cast(Pop AS INT)"))
    .withColumn("age_start", F.expr("try_cast(regexp_extract(AG, '^([0-9]+)', 1) AS INT)")))

seniors = (pop_clean
    .groupBy(F.col("PA").alias("planning_area"), F.col("SZ").alias("subzone"))
    .agg(
        F.sum("pop").alias("residents"),
        F.sum(F.when(F.col("age_start") >= 65, F.col("pop"))).alias("seniors_65plus"),
        F.sum(F.when((F.col("age_start") >= 65) & (F.col("TOD") == "HDB 1- and 2-Room Flats"),
                     F.col("pop"))).alias("seniors_small_flats"))
    .fillna(0, ["seniors_65plus", "seniors_small_flats"]))

(seniors.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.heatcheck.silver_seniors_by_subzone"))

t = seniors.agg(F.sum("seniors_65plus"), F.sum("seniors_small_flats")).first()
print(f"Subzones: {seniors.count()} | Seniors 65+: {t[0]} | In HDB 1-2 room flats: {t[1]}")

In [0]:
import re

def kml_attrs(desc):
    """Pull name/value pairs out of the HTML table in a 'Description' field."""
    return dict(re.findall(r"<th>(.*?)</th>\s*<td>(.*?)</td>", desc or ""))

def centroid(geom):
    """Approximate centre: average of the outline points."""
    polys = geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]
    pts = [pt for poly in polys for pt in poly[0]]
    return sum(p[1] for p in pts) / len(pts), sum(p[0] for p in pts) / len(pts)

sz = json.load(open(f"{FOLDER}/subzones.geojson"))
rows = []
for feat in sz["features"]:
    p = feat["properties"]
    if "SUBZONE_N" not in p:
        p = kml_attrs(p.get("Description"))
    lat, lon = centroid(feat["geometry"])
    rows.append((p["SUBZONE_N"], p["PLN_AREA_N"], lat, lon))

subzones = spark.createDataFrame(rows, "subzone_upper STRING, planning_area_upper STRING, sz_lat DOUBLE, sz_lon DOUBLE")
(subzones.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.heatcheck.silver_subzone_centroids"))
print("Subzones on map:", subzones.count())

In [0]:
elder_rows = []
for feat in eld["features"]:
    a = kml_attrs(feat["properties"].get("Description"))
    lon, lat = feat["geometry"]["coordinates"][:2]
    elder_rows.append((a.get("NAME"), a.get("ADDRESSPOSTALCODE"), lat, lon))

eldercare = spark.createDataFrame(elder_rows, "name STRING, postal_code STRING, lat DOUBLE, lon DOUBLE")
(eldercare.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.heatcheck.silver_eldercare_points"))
print("Eldercare points:", eldercare.count())
display(eldercare.limit(5))

In [0]:
matched = seniors.join(subzones, F.upper(F.trim("subzone")) == F.col("subzone_upper"), "left")
unmatched = matched.filter(F.col("sz_lat").isNull() & (F.col("seniors_65plus") > 0))
print("Subzones with seniors but no map match:", unmatched.count())
display(unmatched.select("planning_area", "subzone", "seniors_65plus"))